In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = None
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping"
wetland_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_Wetland_Shapefiles.gdb"))
census_gdb = os.path.normpath(os.path.join(project_folder, "TIGER_Line_Census.gdb"))

# Project shapefiles

In [2]:
# target coordinate system: NAD 1983 Contiguous USA Albers — WKID 102003
target_sr = arcpy.SpatialReference(102003)

In [3]:
# unedited NWI wetland polygons for illinois
step0 = os.path.normpath(os.path.join(wetland_gdb, "IL_WS_Step0_NWI_Raw"))

In [4]:
# output file for projected polygons
step1 = os.path.normpath(os.path.join(wetland_gdb, "IL_WS_Step1_Project_NAD1983_Albers"))

In [5]:
# implement projection
arcpy.management.Project(
    in_dataset=step0,
    out_dataset=step1,
    out_coor_system=target_sr,
);

In [6]:
# verify output coordinate system
sr = arcpy.Describe(step1).spatialReference
print(f"CRS:   {sr.name}")
print(f"WKID:  {sr.factoryCode}")
print(f"Units: {sr.linearUnitName}")

CRS:   USA_Contiguous_Albers_Equal_Area_Conic
WKID:  102003
Units: Meter


# Clip NWI polygons to Illinois boundary

In [7]:
# Illinois boundary
il_shape_wgs1984 = os.path.normpath(os.path.join(census_gdb, "TIGER_2023_IL_WGS1984"))
il_shape_nad1983_albers = os.path.normpath(os.path.join(census_gdb, "TIGER_2023_IL_NAD1983_Albers"))
arcpy.management.Project(
    in_dataset=il_shape_wgs1984,
    out_dataset=il_shape_nad1983_albers,
    out_coor_system=target_sr,
    transform_method="WGS_1984_(ITRF00)_To_NAD_1983" #"NAD_1983_To_WGS_1984_1"
);

In [8]:
# new clipped NWI polygon shapefile
step2 = os.path.normpath(os.path.join(wetland_gdb, "IL_WS_Step2_Clip"))

In [9]:
# clip NWI polygons to Illinois boundary
arcpy.analysis.PairwiseClip(
    in_features=step1, 
    clip_features=il_shape_nad1983_albers, 
    out_feature_class=step2
);

# Calculate area of wetland polygons

In [10]:
# add fields for both area types
for field, alias in [("Polygon_Area_Ha_Planar","Polygon_Area_Ha_Planar"), ("Polygon_Area_Ha_Geodesic","Polygon_Area_Ha_Geodesic")]:
    if field not in [f.name for f in arcpy.ListFields(step2)]:
        arcpy.management.AddField(step2, field, "DOUBLE")

In [11]:
# calculate PLANAR area
arcpy.management.CalculateGeometryAttributes(
    in_features=step2,
    geometry_property=[["Polygon_Area_Ha_Planar", "AREA"]],
    area_unit="HECTARES",
    coordinate_system=target_sr,  # NAD 1983 Albers
    coordinate_format="SAME_AS_INPUT"                  
);
print("Calculated polygon planar area in hectares.")

Calculated polygon planar area in hectares.


In [12]:
# calculate GEODESIC area
arcpy.management.CalculateGeometryAttributes(
    in_features=step2,
    geometry_property=[["Polygon_Area_Ha_Geodesic", "AREA_GEODESIC"]],
    area_unit="HECTARES",
    coordinate_system=target_sr,
    coordinate_format="SAME_AS_INPUT"
);
print("Calculated polygon geodesic area in hectares.")

Calculated polygon geodesic area in hectares.


# Join NWI cowardin codes

In [13]:
# output shapefile with joined NWI codes
step3 = os.path.normpath(os.path.join(wetland_gdb, "IL_WS_Step3_NWI_Codes"))

In [14]:
# cowardin code definition table
nwi_codes = os.path.normpath(os.path.join(wetland_gdb, "NWI_Code_Definitions_csv"))

In [15]:
# make temporary feature layer 
arcpy.management.MakeFeatureLayer(step2, "temp_layer");

In [16]:
# join table
arcpy.management.AddJoin(
    in_layer_or_view="temp_layer",
    in_field="ATTRIBUTE",       # field in shapefile
    join_table=nwi_codes,
    join_field="ATTRIBUTE",     # field in join table
    join_type="KEEP_ALL"       # KEEP_ALL = left join, KEEP_COMMON = inner join
);

In [17]:
# export joined layer to new shapefile
arcpy.conversion.ExportFeatures(
    in_features="temp_layer",
    out_features=step3
);

In [18]:
# clean up and count
arcpy.management.Delete("temp_layer");
count2 = int(arcpy.management.GetCount(step2)[0])
count3 = int(arcpy.management.GetCount(step3)[0])
print(f"Exported {count2} features in input layer.")
print(f"Exported {count3} features with joined attributes.")

Exported 469190 features in input layer.
Exported 469190 features with joined attributes.


# Select non-deepwater wetlands (not lake or riverine)

In [19]:
# output nondeepwater layer
step4 = os.path.normpath(os.path.join(wetland_gdb, "IL_WS_Step4_NonDeep"))

In [20]:
arcpy.conversion.ExportFeatures(
    in_features = step3,
    out_features = step4,
    where_clause = "WETLAND_TYPE NOT IN ('Lake','Riverine')"
);

In [21]:
# compare polygon count
count3 = int(arcpy.management.GetCount(step3)[0])
count4 = int(arcpy.management.GetCount(step4)[0])
print(f"Previous number of features: {count3}")
print(f"Selected number of features: {count4}")

Previous number of features: 469190
Selected number of features: 312421


In [22]:
# add NWI_ID
arcpy.management.CalculateField(
    in_table=step4,
    field="NWI_ID",
    expression="!OBJECTID!",
    expression_type="PYTHON3"
);